In [1]:
%pip install phonenumbers --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import phonenumbers

Xử lý cho czechia

In [13]:
import pandas as pd

# Đường dẫn tới 2 file
gd_path = r"C:\Users\Nhung\Downloads\We_Love_Pho\sample structure.csv"
checkpoint_path = r"C:\Users\Nhung\Downloads\We_Love_Pho\raw_country_extracted\Slovakia.csv"

# 1. Đọc dữ liệu
df_checkpoint = pd.read_csv(checkpoint_path)
df_gd = pd.read_csv(gd_path)

In [4]:
df_checkpoint.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 544 entries, 0 to 543
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   post_title          544 non-null    object 
 1   address             544 non-null    object 
 2   latitude            544 non-null    float64
 3   longitude           544 non-null    float64
 4   phone               441 non-null    object 
 5   website             247 non-null    object 
 6   facebook            0 non-null      float64
 7   instagram           0 non-null      float64
 8   twitter             0 non-null      float64
 9   post_content        544 non-null    object 
 10  google_maps_link    544 non-null    object 
 11  google_profile      544 non-null    object 
 12  google_review_link  544 non-null    object 
 13  country             544 non-null    object 
dtypes: float64(5), object(9)
memory usage: 59.6+ KB


In [5]:
# --- Kiểm tra định dạng số điện thoại trước khi chuẩn hóa ---
phones = df_checkpoint["phone"].astype(str).str.strip()

# Loại bỏ các nan
valid_phones = phones[~phones.str.lower().isin(["nan", "none", ""]) & (phones != "")]

# Kiểm tra định dạng số điện thoại
# Có bắt đầu bằng '00'?
count_00 = valid_phones.str.startswith("00").sum()
# Có bắt đầu bằng '+' ?
count_plus = valid_phones.str.startswith("+").sum()
# Có dấu '-'?
count_dash = valid_phones.str.contains("-", regex=False).sum()
# Có dấu cách ?
count_space = valid_phones.str.contains(" ", regex=False).sum()

# Kiểm tra range số điện thoại
phones_cleaned = valid_phones.str.replace(r"[-\s]", "", regex=True)
lengths = phones_cleaned.str.len()
min_len = lengths.min()
max_len = lengths.max()

# --- In kết quả ---
print(f"Số bắt đầu bằng '00': {count_00}")
print(f"Số bắt đầu bằng '+': {count_plus}")
print(f"Số chứa dấu '-': {count_dash}")
print(f"Số chứa dấu cách: {count_space}")
print(f"Độ dài ngắn nhất (sau khi loại bỏ ký tự và NaN): {min_len}")
print(f"Độ dài dài nhất: {max_len}")


Số bắt đầu bằng '00': 0
Số bắt đầu bằng '+': 0
Số chứa dấu '-': 0
Số chứa dấu cách: 441
Độ dài ngắn nhất (sau khi loại bỏ ký tự và NaN): 10
Độ dài dài nhất: 11


In [14]:
# Chuẩn hóa số điện thoại theo tiêu chuẩn E.164
from phonenumbers import PhoneNumberFormat
# ---- Country to Region Code Mapping ----
country_region_map = {
    "Sweden": "SE",
    "United Kingdom": "GB",
    "France": "FR",
    "Germany": "DE",
    "Poland": "PL",
    "Czechia": "CZ",
    "Slovakia": "SK",
    "Italy": "IT",
    "Spain": "ES",
    "Portugal": "PT",
    "Belgium": "BE",
    "Netherlands": "NL",
    "Hungary": "HU",
    "Austria": "AT"
}

# Hàm clean - giữ nan và xóa kí tự lạ (gồm dấu cách và - )
def clean_phone_number(raw_phone):
    if pd.isna(raw_phone):
        return raw_phone  
    raw_phone = str(raw_phone)
    cleaned = re.sub(r'[^\d+]', '', raw_phone)  
    return cleaned

# Chuẩn hóa số hợp lệ theo E.164 (thư viện phonenumbers để đưa về dạng sđt quốc tế)
def standardize_phone_number(row):
    raw = clean_phone_number(row["phone"])
    region = country_region_map.get(row["country"], None)
    if pd.isna(raw) or not str(raw).strip():
        return row["phone"]  
    try:
        parsed = phonenumbers.parse(raw, region)
        if phonenumbers.is_valid_number(parsed):
            return phonenumbers.format_number(parsed, PhoneNumberFormat.E164)
        else:
            return row["phone"]
    except:
        return row["phone"]

# Gán nhãn hợp lệ / không hợp lệ / thiếu để tiện lọc thủ công (nếu có)
def label_phone_status(row):
    raw = clean_phone_number(row["phone"])
    region = country_region_map.get(row["country"], None)
    if pd.isna(raw) or not str(raw).strip():
        return pd.NA 
    try:
        parsed = phonenumbers.parse(raw, region)
        if phonenumbers.is_valid_number(parsed):
            return 1  # Valid
        else:
            return 0  # Invalid
    except:
        return 0  # Invalid do lỗi

# Áp dụng hàm
df_checkpoint["phone"] = df_checkpoint.apply(standardize_phone_number, axis=1)
df_checkpoint["phone_status"] = df_checkpoint.apply(label_phone_status, axis=1)
print(df_checkpoint[["phone", "country", "phone_status"]].head(5))

           phone   country phone_status
0  +421950771177  Slovakia            1
1  +421901717516  Slovakia            1
2            NaN  Slovakia         <NA>
3  +421944263935  Slovakia            1
4  +421948845466  Slovakia            1


In [7]:
df_checkpoint.head(5)

,post_title,address,latitude,longitude,phone,website,facebook,instagram,twitter,post_content,google_maps_link,google_profile,google_review_link,country,phone_status
0,Hanoi Garden Restaurant,"1. mája 3866/15, 069 01 Snina, Slovakia",48.990148,22.149454,+421950771177,NaN,NaN,NaN,NaN,Hanoi Garden Restaurant located in 1. mája 386...,https://maps.google.com/?cid=11869844914858063835,https://maps.google.com/?q=place_id:ChIJKUiaZB...,https://search.google.com/local/reviews?placei...,Slovakia,1
1,TAUMI restaurant,"Námestie slobody 57, 066 01 Humenné, Slovakia",48.930944,21.912106,+421901717516,https://www.sushi-rozvoz.sk/restauracia/taumi-...,NaN,NaN,NaN,Online objednávky a rezervácie do podniku Taum...,https://maps.google.com/?cid=5260535499593384612,https://maps.google.com/?q=place_id:ChIJVav44R...,https://search.google.com/local/reviews?placei...,Slovakia,1
2,Pho Yo,"1. mája 31, 031 01 Liptovský Mikuláš, Slovakia",49.082467,19.620736,NaN,NaN,NaN,NaN,NaN,"Pho Yo located in 1. mája 31, 031 01 Liptovský...",https://maps.google.com/?cid=943096168790090850,https://maps.google.com/?q=place_id:ChIJtSTo-T...,https://search.google.com/local/reviews?placei...,Slovakia,<NA>
3,Ha noi fast food & restaurant,"Garbiarska 4300, 031 01 Liptovský Mikuláš, Slo...",49.080795,19.613118,+421944263935,NaN,NaN,NaN,NaN,Ha noi fast food & restaurant located in Garbi...,https://maps.google.com/?cid=5687555765291115668,https://maps.google.com/?q=place_id:ChIJN0BKkd...,https://search.google.com/local/reviews?placei...,Slovakia,1
4,Saigon Asian Restaurant,"Revolučná 162/8, 031 05 Liptovský Mikuláš, Slo...",49.091324,19.599443,+421948845466,http://www.saigonrestaurant.sk/,NaN,NaN,NaN,Saigon Asian Restaurant - Objednať cez interne...,https://maps.google.com/?cid=4513383548255008757,https://maps.google.com/?q=place_id:ChIJ0UGCQS...,https://search.google.com/local/reviews?placei...,Slovakia,1


In [15]:
# tách address thành: street (có zip), zip (postcode), city
def robust_split_address(address, country="Slovakia"):
    if pd.isna(address):
        return "", "", ""

    # Bước 1: Xoá phần country ở cuối
    address = re.sub(
        rf'[,\s]*{re.escape(str(country))}[\s,\d]*$', '', address, flags=re.IGNORECASE
    ).strip()

    # Bước 2: Tìm postcode và city (giả định postcode là 123 45 hoặc 12345 hoặc 123-45)
    match = re.search(r'(\d{3}[-\s]?\d{2})\s+(.+)$', address)
    if match:
        zip_code = match.group(1).strip()
        city = match.group(2).strip()

        # Street là phần trước ZIP (có thể kèm dấu phẩy)
        street_part = address[:match.start()].strip().rstrip(', ')
        street_with_zip = f"{street_part}, {zip_code}"
        return street_with_zip, zip_code, city

    # Nếu không tách được, coi toàn bộ là street
    return address, "", ""

# Áp dụng vào dữ liệu ban đầu
df_checkpoint[['street', 'zip', 'city']] = df_checkpoint['address'].apply(
    lambda x: pd.Series(robust_split_address(x))
)
# xóa cột address
df_checkpoint.drop(columns=['address'], inplace=True)


In [16]:
df_checkpoint.head(5)

,post_title,latitude,longitude,phone,website,facebook,instagram,twitter,post_content,google_maps_link,google_profile,google_review_link,country,phone_status,street,zip,city
0,Hanoi Garden Restaurant,48.990148,22.149454,+421950771177,NaN,NaN,NaN,NaN,Hanoi Garden Restaurant located in 1. mája 386...,https://maps.google.com/?cid=11869844914858063835,https://maps.google.com/?q=place_id:ChIJKUiaZB...,https://search.google.com/local/reviews?placei...,Slovakia,1,"1. mája 3866/15, 069 01",069 01,Snina
1,TAUMI restaurant,48.930944,21.912106,+421901717516,https://www.sushi-rozvoz.sk/restauracia/taumi-...,NaN,NaN,NaN,Online objednávky a rezervácie do podniku Taum...,https://maps.google.com/?cid=5260535499593384612,https://maps.google.com/?q=place_id:ChIJVav44R...,https://search.google.com/local/reviews?placei...,Slovakia,1,"Námestie slobody 57, 066 01",066 01,Humenné
2,Pho Yo,49.082467,19.620736,NaN,NaN,NaN,NaN,NaN,"Pho Yo located in 1. mája 31, 031 01 Liptovský...",https://maps.google.com/?cid=943096168790090850,https://maps.google.com/?q=place_id:ChIJtSTo-T...,https://search.google.com/local/reviews?placei...,Slovakia,<NA>,"1. mája 31, 031 01",031 01,Liptovský Mikuláš
3,Ha noi fast food & restaurant,49.080795,19.613118,+421944263935,NaN,NaN,NaN,NaN,Ha noi fast food & restaurant located in Garbi...,https://maps.google.com/?cid=5687555765291115668,https://maps.google.com/?q=place_id:ChIJN0BKkd...,https://search.google.com/local/reviews?placei...,Slovakia,1,"Garbiarska 4300, 031 01",031 01,Liptovský Mikuláš
4,Saigon Asian Restaurant,49.091324,19.599443,+421948845466,http://www.saigonrestaurant.sk/,NaN,NaN,NaN,Saigon Asian Restaurant - Objednať cez interne...,https://maps.google.com/?cid=4513383548255008757,https://maps.google.com/?q=place_id:ChIJ0UGCQS...,https://search.google.com/local/reviews?placei...,Slovakia,1,"Revolučná 162/8, 031 05",031 05,Liptovský Mikuláš


In [17]:
# Kiểm tra active của web 
import requests
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor

# Chuẩn hóa URL
def normalize_url(url):
    if pd.isna(url) or not str(url).strip():
        return None
    url = url.strip()
    parsed = urlparse(url)
    if not parsed.scheme:
        return "http://" + url
    return url

# Kiểm tra hoạt động website, giữ original URL
def check_url(original_url):
    norm_url = normalize_url(original_url)
    if not norm_url:
        return (original_url, None, None, False)
    try:
        response = requests.get(norm_url, timeout=3, allow_redirects=True)
        final_url = response.url
        status = response.status_code
        is_active = 200 <= status < 400
        return (original_url, status, final_url, is_active)
    except:
        return (original_url, 0, None, False)

# Áp dụng đa luồng
df_checkpoint["normalized_url"] = df_checkpoint["website"].apply(normalize_url)
urls = df_checkpoint["normalized_url"].tolist()

with ThreadPoolExecutor(max_workers=30) as executor:
    results = list(executor.map(check_url, urls))

# Ghi kết quả vào DataFrame
df_checkpoint["web_status"] = [r[1] for r in results]
df_checkpoint["final_url"] = [r[2] for r in results]
df_checkpoint["is_active"] = [r[3] for r in results]


In [18]:
# Kiểm tra các link mạng xã hội bị lẫn trong website
# Xóa cột normalized_url
df_checkpoint.drop(columns=["normalized_url"], inplace=True)

# Xác định nền tảng mạng xã hội 
def classify_social_platform(url):
    if pd.isna(url):
        return None
    url = url.lower()
    if "facebook.com" in url:
        return "facebook"
    elif "instagram.com" in url:
        return "instagram"
    elif "twitter.com" in url or "x.com" in url:
        return "twitter"
    return None

df_checkpoint["social_platform"] = df_checkpoint["website"].apply(classify_social_platform)

# Chuyển các link sai về đúng cột
for platform in ["facebook", "instagram", "twitter"]:
    df_checkpoint[platform] = df_checkpoint.apply(
        lambda row: row["website"] if row["social_platform"] == platform and pd.isna(row[platform]) else row[platform],
        axis=1
    )

# Lưu kết quả vào file CSV
output_path = r"C:\Users\Nhung\Downloads\We_Love_Pho\Clean 25-5 - raw\slovakia_web_check.csv"

In [19]:
# Xoá khỏi giá trị cột website nếu là link MXH 
df_checkpoint.loc[df_checkpoint["social_platform"].notna(), "website"] = None
df_checkpoint.drop(columns=["social_platform"], inplace=True)

In [20]:
df_checkpoint.head(3)


,post_title,latitude,longitude,phone,website,facebook,instagram,twitter,post_content,google_maps_link,google_profile,google_review_link,country,phone_status,street,zip,city,web_status,final_url,is_active
0,Hanoi Garden Restaurant,48.990148,22.149454,+421950771177,NaN,NaN,NaN,NaN,Hanoi Garden Restaurant located in 1. mája 386...,https://maps.google.com/?cid=11869844914858063835,https://maps.google.com/?q=place_id:ChIJKUiaZB...,https://search.google.com/local/reviews?placei...,Slovakia,1,"1. mája 3866/15, 069 01",069 01,Snina,NaN,None,False
1,TAUMI restaurant,48.930944,21.912106,+421901717516,https://www.sushi-rozvoz.sk/restauracia/taumi-...,NaN,NaN,NaN,Online objednávky a rezervácie do podniku Taum...,https://maps.google.com/?cid=5260535499593384612,https://maps.google.com/?q=place_id:ChIJVav44R...,https://search.google.com/local/reviews?placei...,Slovakia,1,"Námestie slobody 57, 066 01",066 01,Humenné,200.0,https://www.sushi-rozvoz.sk/donaska/taumi-asia...,True
2,Pho Yo,49.082467,19.620736,NaN,NaN,NaN,NaN,NaN,"Pho Yo located in 1. mája 31, 031 01 Liptovský...",https://maps.google.com/?cid=943096168790090850,https://maps.google.com/?q=place_id:ChIJtSTo-T...,https://search.google.com/local/reviews?placei...,Slovakia,<NA>,"1. mája 31, 031 01",031 01,Liptovský Mikuláš,NaN,None,False


In [21]:
# tạo copy
df_checkpoint_copy = df_checkpoint.copy()

In [22]:
# 7. Lấy danh sách cột chuẩn từ file gd
gd_columns = df_gd.columns.tolist()

# 8. Thêm các cột còn thiếu và gán giá trị rỗng
for col in gd_columns:
    if col not in df_checkpoint_copy.columns:
        df_checkpoint_copy[col] = ""

# 9. Gán giá trị mặc định
df_checkpoint_copy['post_status'] = 'publish'
df_checkpoint_copy['post_category'] = ',249,'
df_checkpoint_copy['default_category'] = '249'
df_checkpoint_copy['featured'] = '0'

# 10. Sắp xếp lại đúng thứ tự cột
df_checkpoint_copy = df_checkpoint_copy[gd_columns]

# 12. Lưu file đã chuẩn hoá
df_checkpoint_copy.to_csv("slovakia_standardized.csv", index=False)

# 13. (Tuỳ chọn) Xem thử kết quả
print(df_checkpoint_copy[['street', 'zip', 'city', 'country']].head())

                        street     zip               city   country
0      1. mája 3866/15, 069 01  069 01              Snina  Slovakia
1  Námestie slobody 57, 066 01  066 01            Humenné  Slovakia
2           1. mája 31, 031 01  031 01  Liptovský Mikuláš  Slovakia
3      Garbiarska 4300, 031 01  031 01  Liptovský Mikuláš  Slovakia
4      Revolučná 162/8, 031 05  031 05  Liptovský Mikuláš  Slovakia


In [23]:
df_checkpoint_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 544 entries, 0 to 543
Data columns (total 30 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                544 non-null    object 
 1   post_title        544 non-null    object 
 2   post_content      544 non-null    object 
 3   post_status       544 non-null    object 
 4   post_author       544 non-null    object 
 5   post_type         544 non-null    object 
 6   post_date         544 non-null    object 
 7   post_modified     544 non-null    object 
 8   post_tags         544 non-null    object 
 9   post_category     544 non-null    object 
 10  default_category  544 non-null    object 
 11  featured          544 non-null    object 
 12  street            544 non-null    object 
 13  street2           544 non-null    object 
 14  city              544 non-null    object 
 15  region            544 non-null    object 
 16  country           544 non-null    object 
 1

In [24]:
# in ra 5 giá trị đầu cột lat và long
print(df_checkpoint_copy[['latitude', 'longitude']].head())

    latitude  longitude
0  48.990148  22.149454
1  48.930944  21.912106
2  49.082467  19.620736
3  49.080795  19.613118
4  49.091324  19.599443


In [25]:
duplicates = df_checkpoint_copy[df_checkpoint_copy.duplicated(keep=False)]

In [26]:
duplicate_count = duplicates.shape[0]
print(f"Số lượng bản ghi trùng lặp: {duplicate_count}")    


Số lượng bản ghi trùng lặp: 278


In [27]:
# Đếm số lượng giá trị không null cho từng dòng
df_checkpoint_copy['non_null_count'] = df_checkpoint_copy.notnull().sum(axis=1)

# Sắp xếp theo các cột và theo số lượng giá trị không null giảm dần
df_sorted = df_checkpoint_copy.sort_values(by=['post_title', 'latitude', 'longitude', 'street', 'non_null_count'], ascending=[True, True, True, True, False])

# Xóa các dòng trùng hoàn toàn, giữ lại dòng có nhiều thông tin nhất
df_deduplicated = df_sorted.drop_duplicates(keep='first').drop(columns=['non_null_count'])

# Lưu kết quả ra file mới
output_path = "slovakia_no_dup.csv"
df_deduplicated.to_csv(output_path, index=False)


In [28]:
df_deduplicated.info()

<class 'pandas.core.frame.DataFrame'>
Index: 346 entries, 400 to 367
Data columns (total 30 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                346 non-null    object 
 1   post_title        346 non-null    object 
 2   post_content      346 non-null    object 
 3   post_status       346 non-null    object 
 4   post_author       346 non-null    object 
 5   post_type         346 non-null    object 
 6   post_date         346 non-null    object 
 7   post_modified     346 non-null    object 
 8   post_tags         346 non-null    object 
 9   post_category     346 non-null    object 
 10  default_category  346 non-null    object 
 11  featured          346 non-null    object 
 12  street            346 non-null    object 
 13  street2           346 non-null    object 
 14  city              346 non-null    object 
 15  region            346 non-null    object 
 16  country           346 non-null    object 
 17  

Top 200 từ khóa phổ biến và tần suất

In [31]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
# Kết hợp nội dung từ 2 cột post_title và post_content
text_data = df_deduplicated[['post_title']].fillna('').agg(' '.join, axis=1)

# Khởi tạo CountVectorizer để trích xuất từ khóa
vectorizer = CountVectorizer(stop_words='english', max_features=200)
X = vectorizer.fit_transform(text_data)

# Lấy ra từ và tần suất
keywords = vectorizer.get_feature_names_out()
frequencies = np.asarray(X.sum(axis=0)).flatten()

# Tạo dataframe kết quả
keywords_df = pd.DataFrame({'keyword': keywords, 'frequency': frequencies}).sort_values(by='frequency', ascending=False)

In [32]:
# Lưu kết quả vào file CSV
output_keywords_path = "slovakia_keywords.csv"
keywords_df.to_csv(output_keywords_path, index=False)

In [33]:
# Kiểm tra độ chính xác 
positive_keywords = [
    'vietnam', 'viet', 'việt', 'pho', 'bún', 'nem', 'saigon', "phở", 'sài gòn', 'hà nội', 'hanoi', 'bánh mì',
    'halong', 'huế', 'bánh', 'goi cuon', 'bun cha', 'banh', 'thang', 'nam', 'sen', 'hoan kiem', 'wietnam', 'vietnamese',
    'sapa', 'tre', 'ha long', 'ha noi', 'sai gon', 'sajgon', 'hoang', 'ha-noi', 'com tam', 'hải', 'hoan', 'bami',
    'long', 'binh', 'banh mi', 'sao mai', 'song lam', 'ngoc', 'phuong dong', 'linh', 'vietnamská', 'vietnameské', 'quán', 'anh',
    'vietnamskou', 'vietnamské jídlo', 'vietnamská restaurace', 'ngon', 'hoi an', 'quan', 'vina', 'bếp', 'long', 'nón', 'hà', 
    'vietfood', 'gao', 'mì', 'mộc', 'mai', 'thanh', 'cà', 'tuan', 'lá', 'rong', 'vietnamskou', 'vietnamu', 'vietnameske', 'bep', 'hà',
    'huy', 'annam', 'mây', 'huong'
]
negative_keywords = [
    'chinese', 'thai', 'japan', 'korean', 'fusion', 'asia', 'china','india', 'ramen', 'pasta', 'pizza', 'burger',
    'sushi', 'tapas', 'mexican', 'indian', 'kebab', 'italian', 'curry', "tai wan", "singapore", "malaysia", "korea",
    "hong kong", 'resort', 'hotel', 'pub', 'cafe', 'coffee', 'steak', 'park', 'inn', 'post', 'market', 'hall', 'bbq',
    'library', 'sandwich', 'cantonese', 'peking', 'thajská', 'thajské', 'banyan', 'guty', 'shanghai', 'shi', 'pizzerie',
    'bubble', 'kyoto'
]

# Bước 3: Tạo regex pattern
pattern_positive = re.compile('|'.join(positive_keywords), re.IGNORECASE)
pattern_negative = re.compile('|'.join(negative_keywords), re.IGNORECASE)

# Bước 4: Hàm gán tag
def tag_positive(text):
    if pd.isna(text):
        return ''
    return 'Vietnamese restaurant' if pattern_positive.search(text) else ''

def tag_negative(text):
    if pd.isna(text):
        return ''
    return 'Others' if pattern_negative.search(text) else ''

# Bước 5: Gán PositiveTag và NegativeTag
df_deduplicated['Pos'] = df_deduplicated.apply(
    lambda row: tag_positive(row['post_title']) or tag_positive(row['post_content']),
    axis=1
)

df_deduplicated['Neg'] = df_deduplicated.apply(
    lambda row: tag_negative(row['post_title']) or tag_negative(row['post_content']),
    axis=1
)

# Bước 6: Logic gán ReCheck?
def final_recheck_tag(row):
    if row['Pos'] != '' and row['Neg'] == '':
        return 'N'
    elif row['Pos'] == '' and row['Neg'] != '':
        return 'N'
    elif row['Pos'] == '' and row['Neg'] == '':
        return 'Y'
    else:
        return 'Y'

df_deduplicated['ReCheck?'] = df_deduplicated.apply(final_recheck_tag, axis=1)

In [34]:
# Tạo label mẫu
# Tạo 1 cột mới tên là rỗng mới là Y trong df_checkpoint
def assign_label(row):
    pos = row['Pos'] == 'Vietnamese restaurant'
    neg = row['Neg'] == 'Others'
    recheck = row['ReCheck?']

    if pos and not neg and recheck == 'N':
        return 1
    elif neg and not pos and recheck == 'N':
        return 0
    elif pos and neg and recheck == 'Y':
        return 0
    else:
        return ''

df_deduplicated['Y'] = df_deduplicated.apply(assign_label, axis=1)

df = df_deduplicated.copy()

# Xuất file để check manual
columns_to_export = [
    'post_title', 'post_content', 'website', 'google_profile', "Y", 'city'
]
df = df[columns_to_export]
df.to_csv('slovakia_labeled.csv', index=False)


In [35]:
df_deduplicated.info()

<class 'pandas.core.frame.DataFrame'>
Index: 346 entries, 400 to 367
Data columns (total 34 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                346 non-null    object 
 1   post_title        346 non-null    object 
 2   post_content      346 non-null    object 
 3   post_status       346 non-null    object 
 4   post_author       346 non-null    object 
 5   post_type         346 non-null    object 
 6   post_date         346 non-null    object 
 7   post_modified     346 non-null    object 
 8   post_tags         346 non-null    object 
 9   post_category     346 non-null    object 
 10  default_category  346 non-null    object 
 11  featured          346 non-null    object 
 12  street            346 non-null    object 
 13  street2           346 non-null    object 
 14  city              346 non-null    object 
 15  region            346 non-null    object 
 16  country           346 non-null    object 
 17  

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 346 entries, 400 to 367
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   post_title      346 non-null    object
 1   post_content    346 non-null    object
 2   website         118 non-null    object
 3   google_profile  346 non-null    object
 4   Y               346 non-null    object
 5   city            346 non-null    object
dtypes: object(6)
memory usage: 18.9+ KB


In [38]:
# Đọc file labeled
df_labeled = pd.read_csv('slovakia_labeled.csv')
# Kiểm tra số nhãn của từng unique giá trị trong cột 'Y'
print(df_labeled['Y'].value_counts(dropna=False))

Y
1.0    190
0.0     80
NaN     76
Name: count, dtype: int64


In [39]:
print(df_deduplicated['Y'].value_counts(dropna=False))

Y
1    190
0     80
      76
Name: count, dtype: int64


In [40]:
# Kiểm tra lại nhanh số lượng giá trị thiếu (NaN) sau khi chuyển đổi
missing_summary = df_deduplicated.isna().sum()
missing_summary

ID                    0
post_title            0
post_content          0
post_status           0
post_author           0
post_type             0
post_date             0
post_modified         0
post_tags             0
post_category         0
default_category      0
featured              0
street                0
street2               0
city                  0
region                0
country               0
zip                   0
latitude              0
longitude             0
website             228
neighbourhood         0
facebook            304
instagram           342
twitter             346
phone                64
email                 0
logo                  0
google_profile        0
post_images           0
Pos                   0
Neg                   0
ReCheck?              0
Y                     0
dtype: int64

In [41]:
# Chuyển toàn bộ chuỗi rỗng hoặc chuỗi chỉ chứa khoảng trắng thành NaN
df_final = df_deduplicated.copy()

In [42]:
df_final.head(5)

,ID,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,...,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?,Y
400,,.KUKI,"KUKI - Objednať cez internet, zaplatiť v hotov...",publish,,,,,,",249,",...,NaN,+421911803903,,,https://maps.google.com/?q=place_id:ChIJmUMnbY...,,Vietnamese restaurant,,N,1
208,,21,"21 located in Rozmarínová 4249/30, Komárno, 94...",publish,,,,,,",249,",...,NaN,NaN,,,https://maps.google.com/?q=place_id:ChIJ79lz9h...,,,,Y,
408,,36 Asia food & drink,"36 Asia food & drink located in Mierová 64/2, ...",publish,,,,,,",249,",...,NaN,+421911189351,,,https://maps.google.com/?q=place_id:ChIJzbe5E1...,,,Others,N,0
205,,A Pho Bistro,"A Pho Bistro, Bratislava, Slovakia. 8 likes. A...",publish,,,,,,",249,",...,NaN,+421908605999,,,https://maps.google.com/?q=place_id:ChIJrWMQ3D...,,Vietnamese restaurant,,N,1
293,,AN 37 VIETNAM RESTAURANT AND COFFEE,Se menyn hos AN 37 VIETNAM RESTAURANT AND COFF...,publish,,,,,,",249,",...,NaN,NaN,,,https://maps.google.com/?q=place_id:ChIJk4WZPA...,,Vietnamese restaurant,Others,Y,0


In [43]:
# Gán giá trị mặc định nếu thiếu
df_final['post_type'] = df_final['post_type'].fillna('gd_place')
# Gán ngày mặc định nếu thiếu, đảm bảo đúng định dạng chuỗi
default_date = '2025-06-06 00:00:00'
df_final['post_date'] = df_final['post_date'].fillna(default_date)
df_final['post_modified'] = df_final['post_modified'].fillna(default_date)
df_final['post_author'] = df_final['post_author'].fillna('admin')
df_final["post_content"] = ""
df_final = df_final.replace(r'^\s*$', pd.NA, regex=True)
df_final.set_index('ID', inplace=True)
# Đổi tên cột id thành ID
df_final.head(5)
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 346 entries, <NA> to <NA>
Data columns (total 33 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   post_title        346 non-null    object 
 1   post_content      0 non-null      object 
 2   post_status       346 non-null    object 
 3   post_author       0 non-null      object 
 4   post_type         0 non-null      object 
 5   post_date         0 non-null      object 
 6   post_modified     0 non-null      object 
 7   post_tags         0 non-null      object 
 8   post_category     346 non-null    object 
 9   default_category  346 non-null    object 
 10  featured          346 non-null    object 
 11  street            346 non-null    object 
 12  street2           0 non-null      object 
 13  city              346 non-null    object 
 14  region            0 non-null      object 
 15  country           346 non-null    object 
 16  zip               346 non-null    object 
 17

In [44]:
df_final.head(5)
df_final.to_csv('slovakia_final.csv', index=False)

In [45]:
%pip install openpyxl --quiet

# Xuất file excel
output_excel_path = 'slovakia_final.xlsx'
df_final.to_excel(output_excel_path, index=False)

Note: you may need to restart the kernel to use updated packages.


In [55]:
# đọc file labeled_new
df_labeled_new = pd.read_csv('slovakia_labeled_new.csv')
df_standardized= pd.read_csv('slovakia_final.csv')
# Kiểm tra số nhãn của từng unique giá trị trong cột 'Y'
print(df_labeled_new['Y'].value_counts(dropna=False))

Y
1.0    211
NaN     70
0.0     65
Name: count, dtype: int64


In [56]:
# Bổ sung giá trị cột Y từ df_labeled_new vào df_standardized dựa trên index (không có cột ID)
df_standardized['Y'] = df_labeled_new['Y'].values
# Lưu kết quả vào file CSV
output_path = 'slovakia_standardized_with_labels.csv'
# đọc poland_standardized_with_labels.csv
df_standardized.to_csv(output_path, index=False)
df_standardized_with_labels = pd.read_csv(output_path)  
# Kiểm tra số nhãn của từng unique giá trị trong cột 'Y'
print(df_standardized_with_labels['Y'].value_counts(dropna=False))
# Lưu lại file đã chuẩn hoá
df_standardized_with_labels.to_csv('slovakia_standardized_final.csv', index=False)
# in head 5 dòng
df_standardized_with_labels.head(5)

Y
1.0    211
NaN     70
0.0     65
Name: count, dtype: int64


,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,default_category,...,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?,Y
0,.KUKI,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,4.219118e+11,NaN,NaN,https://maps.google.com/?q=place_id:ChIJmUMnbY...,NaN,Vietnamese restaurant,NaN,N,1.0
1,21,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,NaN,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ79lz9h...,NaN,NaN,NaN,Y,NaN
2,36 Asia food & drink,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,4.219112e+11,NaN,NaN,https://maps.google.com/?q=place_id:ChIJzbe5E1...,NaN,NaN,Others,N,0.0
3,A Pho Bistro,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,4.219086e+11,NaN,NaN,https://maps.google.com/?q=place_id:ChIJrWMQ3D...,NaN,Vietnamese restaurant,NaN,N,1.0
4,AN 37 VIETNAM RESTAURANT AND COFFEE,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,NaN,NaN,NaN,https://maps.google.com/?q=place_id:ChIJk4WZPA...,NaN,Vietnamese restaurant,Others,Y,1.0


In [57]:
# Gán giá trị mặc định nếu thiếu
df_standardized_with_labels['post_type'] = df_standardized_with_labels['post_type'].fillna('gd_place')
# Gán ngày mặc định nếu thiếu, đảm bảo đúng định dạng chuỗi
default_date = '2025-06-06 00:00:00'
df_standardized_with_labels['post_date'] = df_standardized_with_labels['post_date'].fillna(default_date)
df_standardized_with_labels['post_modified'] = df_standardized_with_labels['post_modified'].fillna(default_date)
df_standardized_with_labels['post_author'] = df_standardized_with_labels['post_author'].fillna('admin')
df_standardized_with_labels["post_content"] = ""
df_standardized_with_labels = df_standardized_with_labels.replace(r'^\s*$', pd.NA, regex=True)
# trích xuất file cuối chỉ có các dòng mà giá trị cột Y là 1 và xóa cột Y sau đó 
df_standardized_with_labels = df_standardized_with_labels[df_standardized_with_labels['Y'] == 1]
df_standardized_with_labels.drop(columns=['Y'], inplace=True)
df_standardized_with_labels.head(5)

,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,default_category,...,instagram,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?
0,.KUKI,<NA>,publish,admin,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,NaN,",249,",249,...,NaN,NaN,4.219118e+11,NaN,NaN,https://maps.google.com/?q=place_id:ChIJmUMnbY...,NaN,Vietnamese restaurant,NaN,N
3,A Pho Bistro,<NA>,publish,admin,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,NaN,",249,",249,...,NaN,NaN,4.219086e+11,NaN,NaN,https://maps.google.com/?q=place_id:ChIJrWMQ3D...,NaN,Vietnamese restaurant,NaN,N
4,AN 37 VIETNAM RESTAURANT AND COFFEE,<NA>,publish,admin,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,NaN,",249,",249,...,NaN,NaN,NaN,NaN,NaN,https://maps.google.com/?q=place_id:ChIJk4WZPA...,NaN,Vietnamese restaurant,Others,Y
6,Aha Vietnamese coffee & restaurant,<NA>,publish,admin,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,NaN,",249,",249,...,NaN,NaN,4.219406e+11,NaN,NaN,https://maps.google.com/?q=place_id:ChIJXecy-7...,NaN,Vietnamese restaurant,Others,Y
8,Annam Ázijská Reštaurácia,<NA>,publish,admin,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,NaN,",249,",249,...,NaN,NaN,4.219499e+11,NaN,NaN,https://maps.google.com/?q=place_id:ChIJR0FkhE...,NaN,Vietnamese restaurant,NaN,N


In [58]:
# So sánh định dạng từng cột của df_true với df_gd
def compare_column_formats(df1, df2):
    comparison = {}
    for col in df1.columns:
        if col in df2.columns:
            comparison[col] = {
                'df1_dtype': df1[col].dtype,
                'df2_dtype': df2[col].dtype,
                'df1_unique_count': df1[col].nunique(),
                'df2_unique_count': df2[col].nunique()
            }
        else:
            comparison[col] = {
                'df1_dtype': df1[col].dtype,
                'df2_dtype': None,
                'df1_unique_count': df1[col].nunique(),
                'df2_unique_count': None
            }
    return comparison
# So sánh định dạng cột của df_true với df_gd
comparison_result = compare_column_formats(df_standardized_with_labels, df_gd)
# In kết quả so sánh
for col, info in comparison_result.items():
    print(f"Cột: {col}")
    print(f"  - df_true dtype: {info['df1_dtype']}, unique count: {info['df1_unique_count']}")
    print(f"  - df_gd dtype: {info['df2_dtype']}, unique count: {info['df2_unique_count']}")
    print()
# Ép kiểu các cột trong df_true để phù hợp với df_gd
def convert_column_types(df, reference_df):
    for col in reference_df.columns:
        if col in df.columns:
            ref_dtype = reference_df[col].dtype
            if ref_dtype == 'object':
                df[col] = df[col].astype(str)
            elif ref_dtype == 'int64':
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
            elif ref_dtype == 'float64':
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0).astype(float)
            elif ref_dtype == 'datetime64[ns]':
                df[col] = pd.to_datetime(df[col], errors='coerce')
    return df   
# Chuyển đổi kiểu dữ liệu của df_true để phù hợp với df_gd
df_standardized_with_labels = convert_column_types(df_standardized_with_labels, df_gd)

Cột: post_title
  - df_true dtype: object, unique count: 182
  - df_gd dtype: object, unique count: 100

Cột: post_content
  - df_true dtype: object, unique count: 0
  - df_gd dtype: object, unique count: 19

Cột: post_status
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 1

Cột: post_author
  - df_true dtype: object, unique count: 1
  - df_gd dtype: int64, unique count: 19

Cột: post_type
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 1

Cột: post_date
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 100

Cột: post_modified
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 100

Cột: post_tags
  - df_true dtype: float64, unique count: 0
  - df_gd dtype: object, unique count: 1

Cột: post_category
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 1

Cột: default_category
  - df_true dtype: int64, unique count: 1
  - df_gd 

In [59]:
df_standardized_with_labels.head(5)

,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,default_category,...,instagram,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?
0,.KUKI,<NA>,publish,0,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,nan,",249,",249,...,nan,nan,421911803903.0,nan,nan,https://maps.google.com/?q=place_id:ChIJmUMnbY...,0.0,Vietnamese restaurant,NaN,N
3,A Pho Bistro,<NA>,publish,0,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,nan,",249,",249,...,nan,nan,421908605999.0,nan,nan,https://maps.google.com/?q=place_id:ChIJrWMQ3D...,0.0,Vietnamese restaurant,NaN,N
4,AN 37 VIETNAM RESTAURANT AND COFFEE,<NA>,publish,0,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,nan,",249,",249,...,nan,nan,nan,nan,nan,https://maps.google.com/?q=place_id:ChIJk4WZPA...,0.0,Vietnamese restaurant,Others,Y
6,Aha Vietnamese coffee & restaurant,<NA>,publish,0,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,nan,",249,",249,...,nan,nan,421940589988.0,nan,nan,https://maps.google.com/?q=place_id:ChIJXecy-7...,0.0,Vietnamese restaurant,Others,Y
8,Annam Ázijská Reštaurácia,<NA>,publish,0,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,nan,",249,",249,...,nan,nan,421949867057.0,nan,nan,https://maps.google.com/?q=place_id:ChIJR0FkhE...,0.0,Vietnamese restaurant,NaN,N


In [60]:
# Kiểm tra trùng post_title giữa df_standardized_with_labels và df_gd
def check_duplicates(df1, df2, column):
    duplicates = df1[df1[column].isin(df2[column])]
    return duplicates
# Kiểm tra trùng post_title
duplicates = check_duplicates(df_standardized_with_labels, df_gd, 'post_title')
# In ra số lượng bản ghi trùng lặp
print(f"Số lượng bản ghi trùng lặp trong cột 'post_title': {duplicates.shape[0]}")
# In ra 5 bản ghi trùng lặp
print(duplicates[['post_title']].head(5))

Số lượng bản ghi trùng lặp trong cột 'post_title': 1
            post_title
172  PHOčkáreň Eurovea


In [61]:
# Xóa các bản ghi trùng lặp trong df_standardized_with_labels
df_standardized_with_labels = df_standardized_with_labels[~df_standardized_with_labels['post_title'].isin(duplicates['post_title'])]
# In ra số lượng bản ghi sau khi xóa trùng lặp
print(f"Số lượng bản ghi sau khi xóa trùng lặp: {df_standardized_with_labels.shape[0]}")

Số lượng bản ghi sau khi xóa trùng lặp: 210


In [62]:
df_standardized_with_labels.head(5)
#Xóa cột neg, pos, recheck
df_standardized_with_labels.drop(columns=['Neg', 'Pos', 'ReCheck?'], inplace=True)
# In ra thông tin của các cột
df_standardized_with_labels.info()
# Lưu lại file đã chuẩn hoá
df_standardized_with_labels.to_csv('slovakia_upload.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
Index: 210 entries, 0 to 342
Data columns (total 29 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   post_title        210 non-null    object 
 1   post_content      210 non-null    object 
 2   post_status       210 non-null    object 
 3   post_author       210 non-null    int64  
 4   post_type         210 non-null    object 
 5   post_date         210 non-null    object 
 6   post_modified     210 non-null    object 
 7   post_tags         210 non-null    object 
 8   post_category     210 non-null    object 
 9   default_category  210 non-null    int64  
 10  featured          210 non-null    int64  
 11  street            210 non-null    object 
 12  street2           210 non-null    float64
 13  city              210 non-null    object 
 14  region            210 non-null    object 
 15  country           210 non-null    object 
 16  zip               210 non-null    object 
 17  la

In [99]:
import pandas as pd

# 1. Load dữ liệu
df_slovakia = pd.read_csv(r'C:\Users\Nhung\Downloads\We_Love_Pho\slovakia\slovakia_upload.csv')  # hoặc đường dẫn thực tế
df_sample = pd.read_csv(r"C:\Users\Nhung\Downloads\We_Love_Pho\sample structure.csv")

# 2. Thêm cột ID từ index (bắt đầu từ 1)
df_slovakia['ID'] = df_slovakia.index + 1
df_slovakia = df_slovakia[['ID'] + [col for col in df_slovakia.columns if col != 'ID']]

# 3. Đảm bảo các cột đúng thứ tự như file mẫu
df_slovakia = df_slovakia[df_sample.columns]

df_slovakia["post_type"] = "gd_place"
# Gán ngày mặc định nếu thiếu, đảm bảo đúng định dạng chuỗi
df_slovakia['post_date'] = '2025-06-02 00:00:00'
df_slovakia['post_modified'] = '2025-06-02 00:00:00'
df_slovakia['post_author'] = 'admin'

# 4. Chuyển kiểu dữ liệu theo sample
for col in df_sample.columns:
    ref_dtype = df_sample[col].dtype
    if ref_dtype == 'object':
        df_slovakia[col] = df_slovakia[col].astype(str)
    elif 'int' in str(ref_dtype):
        df_slovakia[col] = pd.to_numeric(df_slovakia[col], errors='coerce').fillna(0).astype(int)
    elif 'float' in str(ref_dtype):
        df_slovakia[col] = pd.to_numeric(df_slovakia[col], errors='coerce')
    elif 'datetime' in str(ref_dtype):
        df_slovakia[col] = pd.to_datetime(df_slovakia[col], errors='coerce')

# 5. Làm sạch các chuỗi rỗng hoặc chứa 'nan', 'none'
df_slovakia = df_slovakia.replace(r'^\s*$', pd.NA, regex=True)
df_slovakia = df_slovakia.applymap(lambda x: pd.NA if isinstance(x, str) and x.strip().lower() in ['nan', 'none'] else x)

df_slovakia['region'] = df_slovakia['region'].apply(
    lambda x: "-" if pd.isna(x) or str(x).strip().lower() in ['0.0', 'nan', 'none', 'n/a'] else x
)
# Làm sạch street2 để không có 0.0 hoặc NaN
df_slovakia['street2'] = df_slovakia['street2'].apply(
    lambda x: pd.NA if pd.isna(x) or str(x).strip().lower() in ['0.0', 'nan', 'none'] else x
)

# 6. Chuẩn hóa số điện thoại
df_slovakia['phone'] = df_slovakia['phone'].astype(str).str.replace(r'\.0$', '', regex=True)
df_slovakia['phone'] = df_slovakia['phone'].apply(lambda x: '+' + x if isinstance(x, str) and x and not x.startswith('+') else x)

df_slovakia.drop(columns=['ID'], inplace=True)
# 8. Xuất ra file CSV
df_slovakia.to_csv("slovakia_final_upload_ready.csv", index=False)


C:\Users\Nhung\AppData\Local\Temp\ipykernel_2376\2212474093.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_slovakia = df_slovakia.applymap(lambda x: pd.NA if isinstance(x, str) and x.strip().lower() in ['nan', 'none'] else x)


In [100]:
df_slovakia.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210 entries, 0 to 209
Data columns (total 29 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   post_title        210 non-null    object 
 1   post_content      0 non-null      object 
 2   post_status       210 non-null    object 
 3   post_author       210 non-null    int64  
 4   post_type         210 non-null    object 
 5   post_date         210 non-null    object 
 6   post_modified     210 non-null    object 
 7   post_tags         0 non-null      object 
 8   post_category     210 non-null    object 
 9   default_category  210 non-null    int64  
 10  featured          210 non-null    int64  
 11  street            210 non-null    object 
 12  street2           0 non-null      object 
 13  city              210 non-null    object 
 14  region            210 non-null    object 
 15  country           210 non-null    object 
 16  zip               210 non-null    object 
 1